In [ ]:
# I ran this in Google Colab, was reasonably fast, might take much longer locally.
# This file is just for testing purposes, I haven't bothered with running it locally
# since it also just redownloads the dataset. Also does not make use of the Dataset
# classes that have been implemented, but this notebook is just for finding the means
# and standard deviations anyway.

# imports
import os

import kagglehub
import numpy as np
import pandas as pd
import rasterio
import torch
from torchvision import transforms

In [ ]:
# Paths
path = kagglehub.dataset_download("apollo2506/eurosat-dataset")
RGB_path = os.path.join(path, "EuroSAT")
all_band_path = os.path.join(path, "EuroSATallBands")

Using Colab cache for faster access to the 'eurosat-dataset' dataset.


In [ ]:
# Use pre-existing training set from Kaggle, using the RGB version of the split.
train_path = os.path.join(RGB_path, "train.csv")
train_set = pd.read_csv(train_path)

In [ ]:
# Process all band images in order to find the 2nd and 98th percentile pixel values per band.
# I couldn't find a definitive answer on what values to clip to otherwise. There was a mention
# of a multiplication by 10000 to obtain the current pixel values, but I couldn't find if the
# original range was necessarily [0, 1]. It doesn't seem to be at least.
val_dict = {band: [] for band in range(0, 13)}
for file in train_set['Filename']:
    tif_file = file.replace(".jpg", ".tif")
    file_path = os.path.join(all_band_path, tif_file)
    with rasterio.open(file_path) as src:
        img = src.read().astype(np.float64)
        img = img.reshape(13, -1)
        for i in range(0, 13):
            val_dict[i].append(img[i])

In [ ]:
# Find clipping values per band
clip_values = {}
for band in range(0, 13):
    all_band_pixels = np.concatenate(val_dict[band])
    p2 = np.percentile(all_band_pixels, 2)
    p98 = np.percentile(all_band_pixels, 98)
    clip_values[band] = (p2, p98)
    print(f"Band {band}:")
    print(f"Min: {all_band_pixels.min()}")
    print(f"Max: {all_band_pixels.max()}")
    print("P2 =", p2)
    print("P98 =", p98)

print(clip_values)


Band 0:
Min: 808.0
Max: 17720.0
P2 = 1000.0
P98 = 1964.0
Band 1:
Min: 0.0
Max: 28000.0
P2 = 718.0
P98 = 1961.0
Band 2:
Min: 0.0
Max: 28000.0
P2 = 483.0
P98 = 2071.0
Band 3:
Min: 0.0
Max: 28000.0
P2 = 257.0
P98 = 2528.0
Band 4:
Min: 174.0
Max: 24008.0
P2 = 220.0
P98 = 2587.0
Band 5:
Min: 153.0
Max: 27791.0
P2 = 202.0
P98 = 3680.0
Band 6:
Min: 129.0
Max: 28001.0
P2 = 187.0
P98 = 4579.0
Band 7:
Min: 0.0
Max: 28002.0
P2 = 150.0
P98 = 4531.0
Band 8:
Min: 41.0
Max: 15384.0
P2 = 60.0
P98 = 1702.0
Band 9:
Min: 1.0
Max: 176.0
P2 = 6.0
P98 = 23.0
Band 10:
Min: 5.0
Max: 24704.0
P2 = 27.0
P98 = 3953.0
Band 11:
Min: 1.0
Max: 22210.0
P2 = 15.0
P98 = 2895.0
Band 12:
Min: 91.0
Max: 28000.0
P2 = 140.0
P98 = 5004.0
{0: (np.float64(1000.0), np.float64(1964.0)), 1: (np.float64(718.0), np.float64(1961.0)), 2: (np.float64(483.0), np.float64(2071.0)), 3: (np.float64(257.0), np.float64(2528.0)), 4: (np.float64(220.0), np.float64(2587.0)), 5: (np.float64(202.0), np.float64(3680.0)), 6: (np.float64(187.0), np.f

In [ ]:
# Manually load the 2nd and 98th percentile value if you can't
# be bothered to run the computation from the previous cell.
clip_values = {0: (np.float64(1000.0), np.float64(1964.0)),
               1: (np.float64(718.0), np.float64(1961.0)),
               2: (np.float64(483.0), np.float64(2071.0)),
               3: (np.float64(257.0), np.float64(2528.0)),
               4: (np.float64(220.0), np.float64(2587.0)),
               5: (np.float64(202.0), np.float64(3680.0)),
               6: (np.float64(187.0), np.float64(4579.0)),
               7: (np.float64(150.0), np.float64(4531.0)),
               8: (np.float64(60.0), np.float64(1702.0)),
               9: (np.float64(6.0), np.float64(23.0)),
               10: (np.float64(27.0), np.float64(3953.0)),
               11: (np.float64(15.0), np.float64(2895.0)),
               12: (np.float64(140.0), np.float64(5004.0))}

In [ ]:
# Compute the means and standard deviations of each band
# to use for normalization purposes in the actual project
means = []
stdevs = []
for band in range(0, 13):
    all_band_pixels = np.concatenate(val_dict[band])
    clipped_band = all_band_pixels.clip(min=clip_values[band][0], max=clip_values[band][1])
    means.append(clipped_band.mean())
    stdevs.append(clipped_band.std())

for band in range(13):
    print(f"Band{band} Mean: {means[band]}")
    print(f"Band{band} Stdev: {stdevs[band]}, \n")

Band0 Mean: 1350.6204966130333
Band0 Stdev: 228.87170028962095, 

Band1 Mean: 1108.56131338614
Band1 Stdev: 288.2377114365647, 

Band2 Mean: 1033.6290657422908
Band2 Stdev: 353.65416293463716, 

Band3 Mean: 937.3560028625166
Band3 Stdev: 556.0305161306454, 

Band4 Mean: 1191.3386558185557
Band4 Stdev: 538.2618318700565, 

Band5 Mean: 1996.7863281120824
Band5 Stdev: 846.6016300939549, 

Band6 Mean: 2365.4366610346397
Band6 Stdev: 1066.9121517697647, 

Band7 Mean: 2292.8702064990493
Band7 Stdev: 1099.1289321615109, 

Band8 Mean: 727.79170901021
Band8 Stdev: 391.73180417327205, 

Band9 Mean: 11.951646567046957
Band9 Stdev: 3.758520342333439, 

Band10 Mean: 1814.140218899843
Band10 Stdev: 985.3114209351224, 

Band11 Mean: 1111.522513485863
Band11 Stdev: 738.2927133046666, 

Band12 Mean: 2591.064080429481
Band12 Stdev: 1212.4142852989894, 



In [ ]:
# Manually load the means and standard deviations if you can't
# be bothered to run everything before this (understandable).
means = [np.float64(1350.6204966130333),
         np.float64(1108.56131338614),
         np.float64(1033.6290657422908),
         np.float64(937.3560028625166),
         np.float64(1191.3386558185557),
         np.float64(1996.7863281120824),
         np.float64(2365.4366610346397),
         np.float64(2292.8702064990493),
         np.float64(727.79170901021),
         np.float64(11.951646567046957),
         np.float64(1814.140218899843),
         np.float64(1111.522513485863),
         np.float64(2591.064080429481)]

stdevs = [np.float64(228.87170028962095),
          np.float64(288.2377114365647),
          np.float64(353.65416293463716),
          np.float64(556.0305161306454),
          np.float64(538.2618318700565),
          np.float64(846.6016300939549),
          np.float64(1066.9121517697647),
          np.float64(1099.1289321615109),
          np.float64(391.73180417327205),
          np.float64(3.758520342333439),
          np.float64(985.3114209351224),
          np.float64(738.2927133046666),
          np.float64(1212.4142852989894)]

In [ ]:
# Normalizing functions, will probably have to make these
# integrate into Dataset class in the actual project
def normalize_RGB_img(img: np.ndarray) -> np.ndarray:
    """
    Normalizes a given RGB image to pixel values in the range [0, 1].
    """
    return img / 255

def normalize_all_bands_img(img: np.ndarray, clip_values: dict, means: list, stdevs: list) -> torch.Tensor:
    """
    Normalize all bands of a given image,
    knowing their means and standard deviations
    """
    # First clip per band
    mins = np.array([clip_values[band][0] for band in range(13)])
    maxs = np.array([clip_values[band][1] for band in range(13)])
    img = np.clip(img, mins[:, None, None], maxs[:, None, None])
    tensor = torch.from_numpy(img)

    # Then normalize per band
    normalize = transforms.Normalize(means, stdevs)
    return normalize(tensor)